In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Standard Setup
This cell ensures **reproducibility**, **CUDA detection**, and proper **path management** across all notebooks in this monorepo.

In [ ]:
# ── Standard Setup Cell ──────────────────────────────────────
# Reproducibility | CUDA Detection | Path Management
import os
import random
import logging
from pathlib import Path

import numpy as np

# ── Reproducibility ─────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── PyTorch + CUDA ──────────────────────────────────────────
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"PyTorch {torch.__version__} | Device: {DEVICE}")
    if DEVICE.type == "cuda":
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
except ImportError:
    DEVICE = "cpu"
    print("PyTorch not installed — using CPU")

# ── Paths (relative, portable) ──────────────────────────────
PROJECT_DIR = Path(".").resolve()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project : {PROJECT_DIR.name}")
print(f"Data    : {DATA_DIR}")

# ── Logging ─────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

#  TV Show Recommendation System

In [51]:
#Basic libraries
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from scipy import spatial

import matplotlib.pyplot as plt
import plotly.graph_objects as go

import operator
from sklearn.feature_extraction.text import TfidfVectorizer



import warnings
warnings.filterwarnings('ignore')

In [52]:
df = pd.read_csv(r'C:\Users\79308\Desktop\patel\!Recommendation\11 TV Shows\archive (10)\netflix_titles.csv') #open file using your location

<h1 id="feature" style="font-family:verdana;"> 
    <center>1. Feature Engineering
        <a class="anchor-link" href="https://www.kaggle.com/miljan/interactive-tv-show-recommendation-system/#feature">¶</a>
    </center>
</h1>

In [53]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [54]:
df.shape

(8807, 12)

In [55]:
df['type'].value_counts()

Movie      6131
TV Show    2676
Name: type, dtype: int64

In [56]:
tv_shows = df.loc[df.type=='TV Show'].copy()

In [57]:
tv_shows.shape

(2676, 12)

In [58]:
ratings = pd.read_csv(r'C:\Users\79308\Desktop\patel\!Recommendation\11 TV Shows\archive (11)\TV Shows - Netflix.csv') 

In [59]:
ratings.head()

,Titles,Year,Rating,IMDB_Rating,Netflix
0,Breaking Bad,2008,18+,9.5,1
1,Game of Thrones,2011,18+,9.3,0
2,Rick and Morty,2013,18+,9.2,0
3,Dark,2017,16+,8.8,1
4,Stranger Things,2016,16+,8.8,1


In [60]:
ratings.shape

(50, 5)

In [61]:
ratings = ratings.merge(tv_shows, left_on='Titles', right_on='title')

In [62]:
ratings['Titles']

0                         Breaking Bad
1                                 Dark
2                      Stranger Things
3           Avatar: The Last Airbender
4                             Sherlock
5                              Friends
6                     Better Call Saul
7                         Supernatural
8                         Black Mirror
9                      Attack on Titan
10                      Peaky Blinders
11                The Umbrella Academy
12                              Narcos
13                  Marvel's Daredevil
14                    The Walking Dead
15                Parks and Recreation
16                               Suits
17                            Hannibal
18                              Dexter
19                           Community
20                             Mad Men
21    Fullmetal Alchemist: Brotherhood
22                               Ozark
23                         The Witcher
24                             Lucifer
Name: Titles, dtype: obje

> <h2 id="genre" style="font-family:verdana;"> 
>          1.1 Genre 
>         <a class="anchor-link" href="https://www.kaggle.com/miljan/interactive-movie-recommendation-system/#genre">¶</a>
> 
</h2>

In [63]:
tv_shows.reset_index(inplace=True)

In [64]:
def tag_extractor(feature):
    unique_words = []

    for wordlist in tv_shows[feature].str.split(',').values: 
        for word in wordlist :
            stripped_word = word.lstrip()
            if stripped_word not in unique_words :
                unique_words.append(stripped_word)
                
    return unique_words

In [65]:
genreList = tag_extractor('listed_in')

In [66]:
def binary(x, featureList):
    binaryList = []
    
    for word in featureList:
        
        wList = []
        for w in x.split(','):
            wList.append(w.lstrip())
        
        
        if word in wList:
            binaryList.append(1)
        else:
            binaryList.append(0)
            
    return binaryList

In [67]:
tv_shows['genres_bin'] = tv_shows['listed_in'].apply(lambda x: binary(x, genreList))

> <h2 id="duration" style="font-family:verdana;"> 
>          1.2 Duration 
>         <a class="anchor-link" href="https://www.kaggle.com/miljan/interactive-movie-recommendation-system/#duration">¶</a>
> 
</h2>

In [68]:
import re
tv_shows['duration'] = tv_shows['duration'].apply(lambda x: re.findall(r'\d+', x))

In [69]:
def numb_ex(numb_list):
    for numb in numb_list:
        return int(numb)

In [70]:
tv_shows['duration'] = tv_shows['duration'].apply(lambda x: numb_ex(x))

> <h2 id="rating" style="font-family:verdana;"> 
>          1.3 Rating 
>         <a class="anchor-link" href="https://www.kaggle.com/miljan/interactive-movie-recommendation-system/#rating">¶</a>
> 
</h2>

In [71]:
tv_shows['rating'].value_counts(dropna=False)

TV-MA       1145
TV-14        733
TV-PG        323
TV-Y7        195
TV-Y         176
TV-G          94
NR             5
R              2
NaN            2
TV-Y7-FV       1
Name: rating, dtype: int64

In [72]:
tv_shows.rating = tv_shows.rating.map({'TV-MA': 17, 'TV-14': 14, 'TV-PG': 12, 'TV-G': 10,
                                                  'G': 10, 'TV-Y': 2, 'TV-Y7': 7, 'TV-Y7-FV': 7,
                                                  'NR': 10, 'R': 12, 'NaN': 10, 'PG': 12})

> <h2 id="country" style="font-family:verdana;"> 
>          1.4 Country 
>         <a class="anchor-link" href="https://www.kaggle.com/miljan/interactive-movie-recommendation-system/#country">¶</a>
> 
</h2>

In [73]:
tv_shows['country'] = tv_shows['country'].astype(str)

In [74]:
countryList = tag_extractor('country')

In [75]:
tv_shows['country_bin'] = tv_shows['country'].apply(lambda x: binary(x, countryList))

> <h2 id="casting" style="font-family:verdana;"> 
>          1.5 Casting 
>         <a class="anchor-link" href="https://www.kaggle.com/miljan/interactive-movie-recommendation-system/#casting">¶</a>
> 
</h2>

In [76]:
tv_shows['cast'] = tv_shows['cast'].astype(str)

In [77]:
castList = tag_extractor('cast')

In [78]:
tv_shows['cast_bin'] = tv_shows['cast'].apply(lambda x: binary(x, castList))

***

<h1 id="distance" style="font-family:verdana;"> 
    <center>2. Customized distance function
        <a class="anchor-link" href="https://www.kaggle.com/miljan/interactive-tv-show-recommendation-system/distance">¶</a>
    </center>
</h1>

In [79]:
print('Duplicate entries: {}'.format(tv_shows.duplicated('title').sum()))
tv_shows.drop_duplicates(subset='title', inplace = True)

Duplicate entries: 0


In [80]:
tv_shows['rating'].fillna(10, inplace=True)

In [81]:
tv_shows['description'] = tv_shows['description'].astype(str)

In [82]:
vectorizer = TfidfVectorizer(analyzer = 'word',
                                       min_df=0.0,
                                       max_df = 1.0,
                                       strip_accents = None,
                                       encoding = 'utf-8')

In [83]:
vectorizer.fit_transform(tv_shows['description'].astype(str))

<2676x10378 sparse matrix of type '<class 'numpy.float64'>'
	with 57237 stored elements in Compressed Sparse Row format>

In [84]:
def cosine_sim(text1, text2):
    tfidf = vectorizer.transform([text1, text2])
    return ((tfidf * tfidf.T).A)[0,1]

In [85]:
def distance(show1, show2):
    a = tv_shows.loc[tv_shows.title == show1]
    b = tv_shows.loc[tv_shows.title == show2]
    
    descriptionA = a['description'].values
    descriptionB = b['description'].values
    
    descriptionDistance = 1/cosine_sim(str(descriptionA), str(descriptionB))
    
    
    genresA = a['genres_bin'].values
    genresB = b['genres_bin'].values
    
    dist = 0
    for i in range(0,len(genresA[0])):
        if (genresA[0][i] == genresB[0][i]) and (genresA[0][i]==1):
            dist+=1

    if dist==0:
        genreDistance = len(genresA[0])
        
    else :
        genreDistance = len(genresA[0]) / dist
    
    castA = a['cast_bin'].values
    castB = b['cast_bin'].values
   
    dist = 0
    for i in range(0,len(castA[0])):
        if (castA[0][i] == castB[0][i]) and (castA[0][i]==1):
            dist+=1

    if dist==0:
        castDistance = len(castA[0])
        
    else :
        castDistance = len(castA[0]) / dist

    
    countryA = a['country_bin'].values
    countryB = b['country_bin'].values

    dist = 0
    for i in range(0,len(countryA[0])):
        if (countryA[0][i] == countryB[0][i]) and (countryA[0][i]==1):
            dist+=1

    if dist==0:
        countryDistance = len(countryA[0])
        
    else :
        countryDistance = len(countryA[0]) / dist
    
    ratingA = a['rating'].values
    ratingB = b['rating'].values
    ratingDistance = abs(int(ratingA) - int(ratingB))
                                                   
    durationA = a['duration'].values
    durationB = b['duration'].values
    
    durationDistance = abs(int(durationA) - int(durationB))
                              
                                                   
    return (0.05*descriptionDistance + genreDistance + 0.001*castDistance + 0.05*countryDistance + 0.1*ratingDistance + 0.05*durationDistance)

In [86]:
def get_recommendations(show):

    distances = []
    
    for index, row in tv_shows.iterrows():
        if row['title'] != show:
            dist = distance(row['title'], show)
            distances.append((row['title'], dist))
    
    distances.sort(key=operator.itemgetter(1))
    neighbors = []
    
    for x in range(5):
        neighbors.append(distances[x])

            
    return neighbors

In [87]:
def similar_shows_df(title):
    neighbors = get_recommendations(str(title))
    
    df = tv_shows.loc[tv_shows.title == str(title)][['title', 'listed_in', 'description']]
    
    for i in range(0,5):
        df = df.append(tv_shows.loc[tv_shows.title == neighbors[i][0]][['title', 'listed_in', 'description']], ignore_index=True)
    df['distance'] = 0
    
    dist = [0] + [neighbors[i][1] for i in range(0,5)]
    df['distance'] = dist
    
    return df
    

***

<h1 id="plot" style="font-family:verdana;"> 
    <center>3. Interactive TV Show Recommendations
        <a class="anchor-link" href="https://www.kaggle.com/miljan/interactive-tv-show-recommendation-system/#plot">¶</a>
    </center>
</h1>

In [88]:
ratings['Titles']

0                         Breaking Bad
1                                 Dark
2                      Stranger Things
3           Avatar: The Last Airbender
4                             Sherlock
5                              Friends
6                     Better Call Saul
7                         Supernatural
8                         Black Mirror
9                      Attack on Titan
10                      Peaky Blinders
11                The Umbrella Academy
12                              Narcos
13                  Marvel's Daredevil
14                    The Walking Dead
15                Parks and Recreation
16                               Suits
17                            Hannibal
18                              Dexter
19                           Community
20                             Mad Men
21    Fullmetal Alchemist: Brotherhood
22                               Ozark
23                         The Witcher
24                             Lucifer
Name: Titles, dtype: obje

In [89]:
breaking_bad_df = similar_shows_df('Breaking Bad')
dark_df = similar_shows_df('Dark')
stranger_things_df = similar_shows_df('Stranger Things')
sherlock_df = similar_shows_df('Sherlock')
friends_df =similar_shows_df('Friends')
better_call_saul_df = similar_shows_df('Better Call Saul')
supernatural_df = similar_shows_df('Supernatural')
black_mirror_df = similar_shows_df('Black Mirror')
aot_df = similar_shows_df('Attack on Titan')


In [90]:
peaky_b_df = similar_shows_df('Peaky Blinders')

In [91]:

tua_df = similar_shows_df('The Umbrella Academy')
narcos_df = similar_shows_df('Narcos')
daredevil_df = similar_shows_df("Marvel's Daredevil")
twd_df = similar_shows_df('The Walking Dead')
par_df = similar_shows_df('Parks and Recreation')
suits_df = similar_shows_df('Suits')
dexter_df = similar_shows_df('Dexter')
man_men_df = similar_shows_df('Mad Men')
fma_df = similar_shows_df('Fullmetal Alchemist: Brotherhood')
ozark_df = similar_shows_df('Ozark')
witcher_df = similar_shows_df('The Witcher')
lucifer_df = similar_shows_df('Lucifer')

In [92]:
dfs = [breaking_bad_df, dark_df, stranger_things_df, sherlock_df, friends_df, better_call_saul_df, supernatural_df, black_mirror_df,
      aot_df, peaky_b_df, tua_df, narcos_df, daredevil_df, twd_df, par_df, suits_df,  dexter_df, man_men_df, fma_df, ozark_df, witcher_df, lucifer_df]

In [93]:
data = []

for movie_df in dfs:
    data.append(go.Table(
        header=dict(
            values=[k for k in movie_df.columns],
            font=dict(size=10),
            align="left"
        ),
        cells=dict(
            values=[movie_df[k].tolist() for k in movie_df.columns],
            align = "left")
    ))

In [94]:
fig = go.Figure(data=data)

In [95]:
update_list = []
i = 0
for title in ratings['Titles']:
    update_list.append(dict(label=title, 
                     method="update", 
                     args=[{"visible": ([False]*(i) + [True] + [False]*(len(ratings['Titles'])-i))}, 
                           {"title": title}])
    )
    i+=1

In [96]:
# Add dropdown 
fig.update_layout( 
    updatemenus=[ 
        dict( 
            active=0, 
            buttons=update_list 
        ) 
    ]) 
  